# PAATRA Step 3b — Part 1/5: Setup

Loads the teacher tokenizer, builds the corpus token counts, constructs the three student vocabularies,
tokenizes WikiText into fixed-length chunks, and saves everything to Google Drive.

**Why this exists:** the heavy training step gets split into 3 separate notebooks (one per config). This setup
notebook does the prep work *once* so the three training notebooks can each load the artifacts and run independently
(even in parallel across separate Colab tabs).

**Runtime:** ~3–5 min.

**Output (saved to `MyDrive/paatra/`):**
- `meta.pt` — preset, hyperparameters, special tokens
- `chunks.pt` — tokenized training chunks tensor `[N, SEQ_LEN]`
- `vocabs.pt` — `s2t` / `t2s` mappings for all three student configs


In [ ]:
!pip install -q torch transformers accelerate datasets

In [ ]:
import torch, math, time, gc
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEACHER_ID = "Qwen/Qwen2.5-0.5B"

PRESET = "FULL"   # or "FAST"

if PRESET == "FULL":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 512, 4, 15000, 80_000
elif PRESET == "FAST":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 256, 8, 8000, 50_000

LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.7

print(f'Device: {DEVICE} | Preset: {PRESET}')
print(f'Steps: {NUM_TRAIN_STEPS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}')


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/paatra'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/students', exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')


## 2. Load tokenizer (teacher weights not needed here)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Tokenizer vocab size: {tokenizer.vocab_size:,}')


## 3. Build corpus + count token frequencies

In [ ]:
from collections import Counter
from datasets import load_dataset

print('Loading wikitext-103...')
corpus = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
texts = [t for t in corpus["text"] if len(t.strip()) > 20]
print(f'Filtered texts: {len(texts):,}')

print(f'Counting token frequencies on {NUM_CORPUS_SAMPLES:,} samples...')
token_counts = Counter()
for text in texts[:NUM_CORPUS_SAMPLES]:
    ids = tokenizer.encode(text, add_special_tokens=False)
    token_counts.update(ids)

total_tokens = sum(token_counts.values())
print(f'Total tokens: {total_tokens:,}, unique: {len(token_counts):,}')

special_ids = set()
for attr in ['bos_token_id', 'eos_token_id', 'pad_token_id', 'unk_token_id']:
    tid = getattr(tokenizer, attr, None)
    if tid is not None:
        special_ids.add(tid)
if hasattr(tokenizer, 'additional_special_tokens_ids') and tokenizer.additional_special_tokens_ids:
    special_ids.update(tokenizer.additional_special_tokens_ids)
print(f'Special token ids: {sorted(special_ids)}')


## 4. Build all 3 student vocabularies

In [ ]:
def build_student_vocab(K):
    top_k = {tid for tid, _ in token_counts.most_common(K)}
    keep = sorted(top_k | special_ids)
    t2s = {tid: sid for sid, tid in enumerate(keep)}
    return keep, t2s

STUDENT_CONFIGS = {
    "A_80K_inherited": {"vocab_K": 80_000, "n_embd": 512, "n_layer": 8, "n_head": 8},
    "B_20K_paatra":    {"vocab_K": 20_000, "n_embd": 704, "n_layer": 8, "n_head": 8},
    "C_10K_paatra":    {"vocab_K": 10_000, "n_embd": 768, "n_layer": 8, "n_head": 8},
}

vocabs = {}
print(f"{'Config':<22} {'Vocab':>7}  {'Coverage':>9}  {'Hidden':>7}")
print('─' * 55)
for name, cfg in STUDENT_CONFIGS.items():
    s2t, t2s = build_student_vocab(cfg['vocab_K'])
    covered = sum(c for tid, c in token_counts.items() if tid in t2s)
    vocabs[name] = {'s2t': s2t, 't2s': t2s, 'config': cfg}
    print(f'{name:<22} {len(s2t):>7,}  {covered/total_tokens*100:>8.2f}%  {cfg["n_embd"]:>7}')


## 5. Tokenize corpus into fixed-length chunks

In [ ]:
print('Tokenizing corpus into chunks...')
all_ids = []
for text in texts[:NUM_CORPUS_SAMPLES]:
    all_ids.extend(tokenizer.encode(text, add_special_tokens=False))

n_full = (len(all_ids) - SEQ_LEN) // SEQ_LEN
chunks_list = [all_ids[i*SEQ_LEN:(i+1)*SEQ_LEN] for i in range(n_full)]
chunks = torch.tensor(chunks_list, dtype=torch.long)

print(f'Total tokens: {len(all_ids):,}')
print(f'Chunks tensor: {tuple(chunks.shape)}  ({chunks.element_size() * chunks.nelement() / 1e6:.1f} MB)')
print(f'Effective epochs at {NUM_TRAIN_STEPS} steps, batch {BATCH_SIZE}: {NUM_TRAIN_STEPS * BATCH_SIZE / len(chunks):.2f}')


## 6. Save artifacts to Drive

In [ ]:
meta = {
    'PRESET': PRESET,
    'SEQ_LEN': SEQ_LEN,
    'BATCH_SIZE': BATCH_SIZE,
    'NUM_TRAIN_STEPS': NUM_TRAIN_STEPS,
    'NUM_CORPUS_SAMPLES': NUM_CORPUS_SAMPLES,
    'LEARNING_RATE': LEARNING_RATE,
    'WARMUP_STEPS': WARMUP_STEPS,
    'KD_TEMPERATURE': KD_TEMPERATURE,
    'KD_ALPHA': KD_ALPHA,
    'TEACHER_ID': TEACHER_ID,
    'special_ids': sorted(special_ids),
    'tokenizer_vocab_size': tokenizer.vocab_size,
}

torch.save(meta, f'{DRIVE_DIR}/meta.pt')
torch.save(chunks, f'{DRIVE_DIR}/chunks.pt')
torch.save(vocabs, f'{DRIVE_DIR}/vocabs.pt')

print('Saved to Drive:')
print(f'  {DRIVE_DIR}/meta.pt')
print(f'  {DRIVE_DIR}/chunks.pt')
print(f'  {DRIVE_DIR}/vocabs.pt')
print('\nNext: open 04_train_A.ipynb, 05_train_B.ipynb, or 06_train_C.ipynb.')
